## 1. Install Dependencies

In [1]:
!pip install ultralytics -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 59.7 MB/s eta 0:00:00


## 2. Inspect Dataset Structure

In [2]:
import os

BASE = "/kaggle/input/datasets/valentynsichkar/traffic-signs-dataset-in-yolo-format"
IMG_DIR = f"{BASE}/ts/ts"

all_files = os.listdir(IMG_DIR)
images = sorted([f for f in all_files if f.endswith('.jpg')])
labels = sorted([f for f in all_files if f.endswith('.txt')])

print(f"Total images: {len(images)}")
print(f"Total labels: {len(labels)}")
print(f"Sample images: {images[:5]}")

# Read class names
with open(f"{BASE}/classes.names") as f:
    class_names = [line.strip() for line in f.readlines()]
print(f"Classes: {class_names}")

Total images: 741
Total labels: 741
Sample images: ['00000.jpg', '00001.jpg', '00002.jpg', '00003.jpg', '00004.jpg']
Classes: ['prohibitory', 'danger', 'mandatory', 'other']


## 3. Split Dataset into Train / Val / Test

The dataset has no pre-made split — all images are flat in `ts/ts/`.  
We split 80% train, 10% val, 10% test and copy files into `/kaggle/working/dataset/`.

In [3]:
import shutil
import random
from pathlib import Path

random.seed(42)

# Only keep images that have a matching label file
paired = [img for img in images if img.replace('.jpg', '.txt') in labels]
print(f"Paired image-label samples: {len(paired)}")

random.shuffle(paired)
n = len(paired)
n_train = int(n * 0.8)
n_val   = int(n * 0.1)

splits = {
    "train": paired[:n_train],
    "val":   paired[n_train:n_train + n_val],
    "test":  paired[n_train + n_val:],
}

for split, files in splits.items():
    print(f"{split}: {len(files)} samples")

WORK = Path("/kaggle/working/dataset")

for split, files in splits.items():
    img_out = WORK / "images" / split
    lbl_out = WORK / "labels" / split
    img_out.mkdir(parents=True, exist_ok=True)
    lbl_out.mkdir(parents=True, exist_ok=True)

    for img_name in files:
        lbl_name = img_name.replace('.jpg', '.txt')
        shutil.copy(f"{IMG_DIR}/{img_name}", img_out / img_name)
        shutil.copy(f"{IMG_DIR}/{lbl_name}", lbl_out / lbl_name)

print("Dataset split and copied successfully!")

Paired image-label samples: 741
train: 592 samples
val: 74 samples
test: 75 samples
Dataset split and copied successfully!


## 4. Create `dataset.yaml`

In [4]:
import yaml

dataset_yaml = {
    "path": "/kaggle/working/dataset",
    "train": "images/train",
    "val":   "images/val",
    "test":  "images/test",
    "nc": len(class_names),
    "names": class_names,
}

yaml_path = "/kaggle/working/dataset.yaml"
with open(yaml_path, "w") as f:
    yaml.dump(dataset_yaml, f, default_flow_style=False)

print("dataset.yaml contents:")
with open(yaml_path) as f:
    print(f.read())

dataset.yaml contents:
names:
- prohibitory
- danger
- mandatory
- other
nc: 4
path: /kaggle/working/dataset
test: images/test
train: images/train
val: images/val



## 5. Train YOLOv8

In [5]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")  # n=nano (fastest). Options: n, s, m, l, x

results = model.train(
    data=yaml_path,
    epochs=50,
    imgsz=640,
    batch=16,
    lr0=0.01,
    patience=15,
    project="/kaggle/working/weights",
    name="yolo_traffic",
    exist_ok=True,
    device=0,        # GPU
    val=True,
    plots=True,
    save=True,
    verbose=True,
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.48 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=

## 6. Evaluate on Test Set

In [6]:
best_weights = "/kaggle/working/weights/yolo_traffic/weights/best.pt"
model = YOLO(best_weights)

metrics = model.val(
    data=yaml_path,
    split="test",
    imgsz=640,
    device=0,
    verbose=True,
)

print("\n--- YOLOv8 Test Evaluation ---")
print(f"mAP50:     {metrics.box.map50:.4f}")
print(f"mAP50-95:  {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall:    {metrics.box.mr:.4f}")

Ultralytics 8.4.48 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,006,428 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2954.6±903.4 MB/s, size: 290.7 KB)
val: Scanning /kaggle/working/dataset/labels/test... 75 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 75/75 1.3Kit/s 0.1s
val: New cache created: /kaggle/working/dataset/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 5/5 2.8it/s 1.8s0.5s
                   all         75        139      0.947      0.951      0.969      0.806
           prohibitory         38         58      0.986      0.983      0.995      0.867
                danger         20         26      0.942          1      0.994      0.839
             mandatory         20         24          1      0.951      0.955      0.783
                 other         25         31      0.859      0.871      0